In [ ]:
# Computer Vision Engineer
#
# This project incorporates components from the Apache 2.0 licensed project.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# ******************************************************************************
# DISCLAIMER:
#
# This notebook downloads images and bounding-box annotations from Google Open
# Images. Those files are covered by their own licenses. Read the terms at
# https://storage.googleapis.com/openimages/web/index.html and follow them
# for every image and annotation this notebook saves.
#
# By using this notebook, you agree to those terms. Use that breaks them is
# your responsibility.
# ******************************************************************************
#
# Edit classes, out_dir, and max_number_images_per_class in the download cell,
# then run the cells from the top. Class names must match the Open Images
# DisplayName column exactly, including capitalization.
#
# The class table is the v7 file. Train boxes are the v6 file. Validation and
# test boxes are the v5 files. Those files and downloader.py are saved in the
# current directory and reused when they are already there.
#
# JPEGs are fetched into ./.out, which is deleted at the end. With
# yolov8_format True, the dataset is written as:
#
#   <out_dir>/images/train|val|test/
#   <out_dir>/labels/train|val|test/
#
# The Open Images validation split is saved under val. Each label line is
# class_id x_center y_center width height, with values from 0 to 1. Class 0 is
# the first name in classes.
#
# Each image id is queued once and counts toward the first requested class
# found on that image. This notebook queues up to 200 ids per class.
#
# The zip and Google Drive cells expect a Colab runtime, where ./data is
# /content/data.

In [ ]:
# pandas and requests are imported by the cells below.
# The fetched downloader.py also imports requests and tqdm.
# boto3 and opencv-python are installed here and are not imported by this notebook.

!pip install boto3
!pip install tqdm
!pip install requests
!pip install pandas
!pip install opencv-python

In [2]:
import os
import shutil
import sys

import pandas as pd
import requests


def process(classes, data_out_dir, yolov8_format, max_number_images_per_class):
    """Download Open Images boxes for the given display names.

    Names must match DisplayName exactly. List order becomes the YOLO class
    ids, starting at 0. None for the image cap means no limit. The validation
    split is written under val.
    """
    if max_number_images_per_class is None:
        max_number_images_per_class = sys.maxsize

    # Class names: Open Images v7. Train boxes: v6. Validation and test boxes: v5.
    train_data_url = (
        "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv"
    )
    val_data_url = (
        "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv"
    )
    test_data_url = (
        "https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv"
    )

    downloader_url = (
        "https://raw.githubusercontent.com/openimages/dataset/master/downloader.py"
    )

    class_names_all_url = (
        "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv"
    )

    # Save each table, and downloader.py, in the current directory.
    # Skip a file that is already there.
    for url in [
        train_data_url,
        val_data_url,
        test_data_url,
        class_names_all_url,
        downloader_url,
    ]:
        if not os.path.exists(url.split("/")[-1]):
            print("downloading {}...".format(url.split("/")[-1]))
            r = requests.get(url)
            with open(url.split("/")[-1], "wb") as f:
                f.write(r.content)

    class_ids = []

    classes_all = pd.read_csv(class_names_all_url.split("/")[-1])

    # DisplayName match is exact. An unknown name stops the run.
    for class_ in classes:
        if class_ not in list(classes_all["DisplayName"]) or class_ not in list(
            classes_all["DisplayName"]
        ):
            raise Exception("Class name not found: {}".format(class_))
        class_index = list(classes_all["DisplayName"]).index(class_)
        class_ids.append(classes_all["LabelName"].iloc[class_index])

    image_list_file_path = os.path.join(".", "image_list_file")
    if os.path.exists(image_list_file_path):
        os.remove(image_list_file_path)

    # Queue each image id once. It counts toward the first requested class on that image.
    # Lines are train|validation|test/<id> for downloader.py.
    image_list_file_list = []
    for j, url in enumerate([train_data_url, val_data_url, test_data_url]):
        image_list_file_per_class = [[] for j in class_ids]
        filename = url.split("/")[-1]
        with open(filename, "r") as f:
            line = f.readline()
            while len(line) != 0:
                id, _, class_name, _, x1, x2, y1, y2, _, _, _, _, _ = line.split(",")[
                    :13
                ]
                if (
                    class_name in class_ids
                    and id not in image_list_file_list
                    and len(image_list_file_per_class[class_ids.index(class_name)])
                    < max_number_images_per_class
                ):
                    image_list_file_list.append(id)
                    image_list_file_per_class[class_ids.index(class_name)].append(id)
                    with open(image_list_file_path, "a") as fw:
                        fw.write(
                            "{}/{}\n".format(["train", "validation", "test"][j], id)
                        )
                line = f.readline()

            f.close()

    # Fetch the JPEGs with the Open Images downloader. python must be on PATH.
    out_dir = "./.out"
    shutil.rmtree(out_dir, ignore_errors=True)
    os.system(
        "python downloader.py {} --download_folder={}".format(
            image_list_file_path, out_dir
        )
    )

    DATA_ALL_DIR = out_dir

    # Replace train, val, and test under data_out_dir. Other files there stay.
    for set_ in ["train", "val", "test"]:
        for dir_ in [
            os.path.join(data_out_dir, set_),
            os.path.join(data_out_dir, set_, "imgs"),
            os.path.join(data_out_dir, set_, "anns"),
        ]:
            if os.path.exists(dir_):
                shutil.rmtree(dir_)
            os.makedirs(dir_)

    for j, url in enumerate([train_data_url, val_data_url, test_data_url]):
        filename = url.split("/")[-1]
        set_ = ["train", "val", "test"][j]
        print(filename)
        with open(filename, "r") as f:
            line = f.readline()
            while len(line) != 0:
                id, _, class_name, _, x1, x2, y1, y2, _, _, _, _, _ = line.split(",")[
                    :13
                ]
                if class_name in class_ids:
                    if os.path.exists(os.path.join(DATA_ALL_DIR, "{}.jpg".format(id))):
                        if not os.path.exists(
                            os.path.join(
                                data_out_dir, set_, "imgs", "{}.jpg".format(id)
                            )
                        ):
                            shutil.copy(
                                os.path.join(DATA_ALL_DIR, "{}.jpg".format(id)),
                                os.path.join(
                                    data_out_dir, set_, "imgs", "{}.jpg".format(id)
                                ),
                            )
                        with open(
                            os.path.join(
                                data_out_dir, set_, "anns", "{}.txt".format(id)
                            ),
                            "a",
                        ) as f_ann:
                            # class_id x_center y_center width height, values from 0 to 1
                            x1, x2, y1, y2 = [float(j) for j in [x1, x2, y1, y2]]
                            xc = (x1 + x2) / 2
                            yc = (y1 + y2) / 2
                            w = x2 - x1
                            h = y2 - y1

                            f_ann.write(
                                "{} {} {} {} {}\n".format(
                                    int(class_ids.index(class_name)), xc, yc, w, h
                                )
                            )
                            f_ann.close()

                line = f.readline()

    shutil.rmtree(out_dir, ignore_errors=True)

    # Copy into images/<split> and labels/<split>, then remove the imgs/anns folders.
    if yolov8_format:
        for set_ in ["train", "val", "test"]:
            for dir_ in [
                os.path.join(data_out_dir, "images", set_),
                os.path.join(data_out_dir, "labels", set_),
            ]:
                if os.path.exists(dir_):
                    shutil.rmtree(dir_)
                os.makedirs(dir_)

            for filename in os.listdir(os.path.join(data_out_dir, set_, "imgs")):
                shutil.copy(
                    os.path.join(data_out_dir, set_, "imgs", filename),
                    os.path.join(data_out_dir, "images", set_, filename),
                )
            for filename in os.listdir(os.path.join(data_out_dir, set_, "anns")):
                shutil.copy(
                    os.path.join(data_out_dir, set_, "anns", filename),
                    os.path.join(data_out_dir, "labels", set_, filename),
                )

            shutil.rmtree(os.path.join(data_out_dir, set_))

In [ ]:
# Open Images display names, in the order that becomes YOLO class ids.
# Matching is exact, including capitalization. The usual table spellings are
# Person, Handgun, Shotgun, Rifle, and Knife.
classes = [
    "person",
    "handgun",
    "shotgun",
    "rifle",
    "knife",
]

# Dataset root. Existing train, val, and test folders under it are replaced.
out_dir = "./data"

# Image ids to queue for each class. An id is queued once, toward the first
# requested class found on that image.
max_number_images_per_class = 200

# True writes images/<split> and labels/<split>, then removes the imgs/anns folders.
# The Open Images validation split is saved under val.
yolov8_format = True

process(classes, out_dir, yolov8_format, max_number_images_per_class)

downloading oidv6-train-annotations-bbox.csv...
downloading validation-annotations-bbox.csv...
downloading test-annotations-bbox.csv...
downloading oidv7-class-descriptions.csv...
downloading downloader.py...
oidv6-train-annotations-bbox.csv
validation-annotations-bbox.csv
test-annotations-bbox.csv


In [ ]:
# Colab only. Zip /content/data, which is ./data when the notebook runs from /content,
# into data.zip in the current directory.

!zip -r data.zip /content/data

In [ ]:
# Colab only. Mount Google Drive at /content/gdrive.

from google.colab import drive

drive.mount("/content/gdrive")

In [6]:
# Colab only, after the Drive mount and the zip cell. Copy data.zip to
# My Drive/data.zip on the mounted drive. Both paths are on this machine.

!scp '/content/data.zip' '/content/gdrive/My Drive/data.zip'